# E1 — Canonical Reproduction & Evidence Lock
## Calib-BERT v2 Reconstruction

**Purpose:** reproduce the canonical APIN-D-26-10636 experiment *before* adding new methods, backbones, or distribution-shift experiments.

### Canonical facts from the submitted manuscript
- Dreaddit development = **2,838**, official test = **715**
- Development split by `post_id` through **20 stratified group folds**
  - folds 0–13: fit = **1,994**
  - folds 14–16: validation = **421**
  - folds 17–19: calibration = **423**
- split seed = **42**
- BERT-base-uncased
- max length = **128**
- learning rate = **2e-5**
- train/eval batch = **16/32**
- weight decay = **0.01**
- epochs = **3**
- AdamW + linear scheduler, zero warm-up, gradient clipping = **1.0**
- primary seed = **42**
- repeated seeds = **13, 21, 42, 87, 100**
- selection rule: lowest validation ECE → lowest Brier → highest F1
- selected candidate in the submitted study: **LS ε=0.10**
- temperature scaling is fitted only on the untouched calibration subset
- official test is evaluated after model-selection lock

### Reproducibility warning
The manuscript does not fully specify the raw filenames/schema, exact splitting API, or the numerical `a_min` used in AALS. Therefore this notebook:
1. prefers the **original split manifest** if available;
2. otherwise attempts a *candidate* `StratifiedGroupKFold` reconstruction and refuses to proceed if the manuscript counts do not match;
3. keeps AALS disabled until the original agreement field and `a_min` are recovered.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/calibbert_e1")
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "e1"
RESULT_DIR = PROJECT_ROOT / "results"
MODEL_DIR = PROJECT_ROOT / "models"

for p in [DATA_DIR, ARTIFACT_DIR, RESULT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEV_PATH = DATA_DIR / "dreaddit_dev.csv"
TEST_PATH = DATA_DIR / "dreaddit_test.csv"
CANONICAL_SPLIT_MANIFEST = DATA_DIR / "split_manifest.csv"

ID_COL = "id"
POST_COL = "post_id"
TEXT_COL = "text"
LABEL_COL = "label"
LABEL_MAP = None

AGREEMENT_COL = None
A_MIN = None

RUN_AALS = False
RUN_MULTISEED = False
DEVICE_PREFERENCE = "cuda"
REPRO_ABS_TOL = 0.005


## 1. Environment audit

The submitted manuscript reports Python 3.12.13, PyTorch 2.11.0 + CUDA 12.8, Transformers 5.13.1, scikit-learn 1.6.1, and Matplotlib 3.10.8.

For strict reproduction, prefer the original environment if available. This notebook records the active environment rather than silently replacing it.


In [ ]:
import sys, json, platform, os, random, hashlib, warnings
import numpy as np
import pandas as pd

def package_version(name):
    try:
        import importlib.metadata as md
        return md.version(name)
    except Exception:
        return "NOT_INSTALLED"

env = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "torch": package_version("torch"),
    "transformers": package_version("transformers"),
    "scikit-learn": package_version("scikit-learn"),
    "scipy": package_version("scipy"),
    "matplotlib": package_version("matplotlib"),
}
print(json.dumps(env, indent=2))
(ARTIFACT_DIR / "environment.json").write_text(json.dumps(env, indent=2))


In [ ]:
# Optional strict-environment install cell.
# Uncomment only if the exact reported versions are available in your runtime.
#
# %pip install -q "torch==2.11.0" "transformers==5.13.1" \
#     "scikit-learn==1.6.1" "scipy" "matplotlib==3.10.8" "pandas" "numpy"


## 2. Imports and deterministic setup

In [ ]:
import math, copy
from dataclasses import dataclass, asdict

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, confusion_matrix
from scipy.special import softmax
from scipy.optimize import minimize_scalar
from scipy.stats import binomtest

PRIMARY_SEED = 42
SPLIT_SEED = 42
PAIRED_SEEDS = [13, 21, 42, 87, 100]

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(PRIMARY_SEED)
device = torch.device("cuda" if DEVICE_PREFERENCE == "cuda" and torch.cuda.is_available() else "cpu")
print("Device:", device)


## 3. Load and audit the exact Dreaddit input files

In [ ]:
def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Upload/copy the exact file used in the APIN experiment.")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

dev = read_table(DEV_PATH)
test = read_table(TEST_PATH)

required = [ID_COL, POST_COL, TEXT_COL, LABEL_COL]
for name, df in [("dev", dev), ("test", test)]:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"{name} missing required columns: {missing}")

if LABEL_MAP is not None:
    dev[LABEL_COL] = dev[LABEL_COL].map(LABEL_MAP)
    test[LABEL_COL] = test[LABEL_COL].map(LABEL_MAP)

for name, df in [("dev", dev), ("test", test)]:
    if df[LABEL_COL].isna().any():
        raise ValueError(f"{name}: missing/unmapped labels detected")
    uniq = sorted(pd.unique(df[LABEL_COL]).tolist())
    if uniq != [0, 1]:
        raise ValueError(f"{name}: expected labels [0,1], found {uniq}")

print("Development:", dev.shape)
print("Official test:", test.shape)
print("Dev labels:\n", dev[LABEL_COL].value_counts().sort_index())
print("Test labels:\n", test[LABEL_COL].value_counts().sort_index())

assert len(dev) == 2838, f"Expected dev=2838, found {len(dev)}"
assert len(test) == 715, f"Expected test=715, found {len(test)}"


## 4. Hash raw inputs and audit the official split

Submitted manuscript targets:
- `id` overlap = 0
- `post_id` overlap = 0
- exact cross-split text duplicates = 3


In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

input_hashes = {
    "dev_path": str(DEV_PATH),
    "dev_sha256": sha256_file(DEV_PATH),
    "test_path": str(TEST_PATH),
    "test_sha256": sha256_file(TEST_PATH),
}
(ARTIFACT_DIR / "input_hashes.json").write_text(json.dumps(input_hashes, indent=2))
print(json.dumps(input_hashes, indent=2))

id_overlap = set(dev[ID_COL].astype(str)) & set(test[ID_COL].astype(str))
post_overlap = set(dev[POST_COL].astype(str)) & set(test[POST_COL].astype(str))

dup_pairs = dev[[ID_COL, POST_COL, TEXT_COL]].merge(
    test[[ID_COL, POST_COL, TEXT_COL]],
    on=TEXT_COL,
    suffixes=("_dev", "_test")
)

print("ID overlap:", len(id_overlap))
print("post_id overlap:", len(post_overlap))
print("Exact text duplicate pairs:", len(dup_pairs))

assert len(id_overlap) == 0
assert len(post_overlap) == 0
dup_pairs.to_csv(ARTIFACT_DIR / "duplicate_audit.csv", index=False)

if len(dup_pairs) != 3:
    warnings.warn(f"Manuscript reports 3 exact text duplicates; current data produce {len(dup_pairs)}.")


## 5. Recover exact fit / validation / calibration split

Preferred: original `split_manifest.csv` with `id` and `subset`.

Fallback: a candidate reconstruction using `StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)`. This fallback is explicitly an implementation assumption and must match manuscript counts before proceeding.


In [ ]:
EXPECTED_SUBSET_COUNTS = {"fit": 1994, "validation": 421, "calibration": 423}
EXPECTED_CLASS_COUNTS = {
    "fit": {0: 942, 1: 1052},
    "validation": {0: 203, 1: 218},
    "calibration": {0: 205, 1: 218},
}

def validate_manifest(manifest, dev_df):
    if ID_COL not in manifest.columns or "subset" not in manifest.columns:
        raise KeyError("Manifest must contain id and subset columns")

    allowed = {"fit", "validation", "calibration"}
    bad = set(manifest["subset"].unique()) - allowed
    if bad:
        raise ValueError(f"Unexpected subset labels: {bad}")

    m = manifest[[ID_COL, "subset"]].copy()
    m[ID_COL] = m[ID_COL].astype(str)
    d = dev_df.copy()
    d[ID_COL] = d[ID_COL].astype(str)

    if set(m[ID_COL]) != set(d[ID_COL]):
        raise ValueError("Manifest IDs do not match development IDs")

    merged = d.merge(m, on=ID_COL, how="left", validate="one_to_one")

    got_counts = merged["subset"].value_counts().to_dict()
    print("Subset counts:", got_counts)
    for subset, expected in EXPECTED_SUBSET_COUNTS.items():
        assert got_counts.get(subset, 0) == expected, (subset, got_counts.get(subset,0), expected)

    for subset, expected in EXPECTED_CLASS_COUNTS.items():
        got = merged.loc[merged["subset"] == subset, LABEL_COL].value_counts().sort_index().to_dict()
        print(subset, "class counts:", got)
        assert got == expected, (subset, got, expected)
    return merged

if CANONICAL_SPLIT_MANIFEST.exists():
    print("Using original split manifest:", CANONICAL_SPLIT_MANIFEST)
    manifest = pd.read_csv(CANONICAL_SPLIT_MANIFEST)
    dev_split = validate_manifest(manifest, dev)
    split_source = "original_manifest"
else:
    print("No original manifest found. Building candidate reconstruction.")
    X = np.zeros(len(dev))
    y = dev[LABEL_COL].to_numpy()
    groups = dev[POST_COL].astype(str).to_numpy()
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SPLIT_SEED)

    fold_id = np.full(len(dev), -1, dtype=int)
    for fold, (_, idx) in enumerate(sgkf.split(X, y, groups)):
        fold_id[idx] = fold

    candidate = dev[[ID_COL]].copy()
    candidate["fold"] = fold_id
    candidate["subset"] = np.select(
        [
            candidate["fold"].between(0, 13),
            candidate["fold"].between(14, 16),
            candidate["fold"].between(17, 19),
        ],
        ["fit", "validation", "calibration"],
        default="ERROR"
    )
    candidate.to_csv(ARTIFACT_DIR / "candidate_split_manifest.csv", index=False)

    try:
        dev_split = validate_manifest(candidate[[ID_COL, "subset"]], dev)
        split_source = "candidate_StratifiedGroupKFold"
        print("Candidate matches manuscript counts. Still compare IDs with the original manifest if available.")
    except Exception as e:
        print("Candidate reconstruction failed:", repr(e))
        raise RuntimeError("Recover the original split manifest/splitting code before proceeding.")

working_manifest = dev_split[[ID_COL, POST_COL, LABEL_COL, "subset"]].copy()
working_manifest.to_csv(ARTIFACT_DIR / "split_manifest_working.csv", index=False)
(ARTIFACT_DIR / "split_source.txt").write_text(split_source)

fit_df = dev_split.query("subset == 'fit'").reset_index(drop=True)
val_df = dev_split.query("subset == 'validation'").reset_index(drop=True)
cal_df = dev_split.query("subset == 'calibration'").reset_index(drop=True)

print(len(fit_df), len(val_df), len(cal_df))


## 6. Canonical model/training configuration

In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    model_name: str = "bert-base-uncased"
    max_length: int = 128
    learning_rate: float = 2e-5
    train_batch_size: int = 16
    eval_batch_size: int = 32
    weight_decay: float = 0.01
    epochs: int = 3
    warmup_steps: int = 0
    max_grad_norm: float = 1.0

CFG = TrainConfig()
print(asdict(CFG))
(ARTIFACT_DIR / "train_config.json").write_text(json.dumps(asdict(CFG), indent=2))
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)


## 7. Dataset and manuscript-faithful loss functions

The notebook implements the manuscript equations directly:
- fixed LS: true target = `1-ε`, non-true = `ε/(K-1)`
- AALS: `ε_i = ε(1-a_i)/(1-a_min)`

AALS remains disabled until the original agreement field and `a_min` are recovered.


In [ ]:
class TextDataset(Dataset):
    def __init__(self, df, include_agreement=False):
        self.df = df.reset_index(drop=True)
        self.enc = tokenizer(
            self.df[TEXT_COL].astype(str).tolist(),
            truncation=True,
            padding=False,
            max_length=CFG.max_length,
        )
        self.labels = self.df[LABEL_COL].astype(int).to_numpy()
        self.include_agreement = include_agreement
        if include_agreement:
            if AGREEMENT_COL is None or AGREEMENT_COL not in self.df.columns:
                raise ValueError("AALS requested but agreement field is not configured")
            self.agreement = self.df[AGREEMENT_COL].astype(float).to_numpy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.include_agreement:
            item["agreement"] = torch.tensor(self.agreement[idx], dtype=torch.float32)
        return item

def collate_batch(batch):
    labels = torch.stack([x.pop("labels") for x in batch])
    agreement = None
    if "agreement" in batch[0]:
        agreement = torch.stack([x.pop("agreement") for x in batch])
    padded = tokenizer.pad(batch, padding=True, return_tensors="pt")
    padded["labels"] = labels
    if agreement is not None:
        padded["agreement"] = agreement
    return padded

def soft_target_cross_entropy(logits, labels, epsilon):
    K = logits.shape[-1]
    q = torch.full_like(logits, fill_value=epsilon / (K - 1))
    q.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)
    return -(q * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

def aals_cross_entropy(logits, labels, agreement, epsilon_max, a_min):
    if a_min is None:
        raise ValueError("A_MIN is unknown; recover it from the original experiment")
    eps_i = epsilon_max * (1.0 - agreement) / (1.0 - a_min)
    eps_i = eps_i.clamp(min=0.0, max=epsilon_max)
    K = logits.shape[-1]
    q = eps_i.unsqueeze(1).repeat(1, K) / (K - 1)
    q.scatter_(1, labels.unsqueeze(1), (1.0 - eps_i).unsqueeze(1))
    return -(q * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()


## 8. Canonical evaluation metrics

In [ ]:
def ece_equal_width(y_true, probs, n_bins=10):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = 0.0
    bins_out = []
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (conf >= lo) & ((conf <= hi) if b == n_bins-1 else (conf < hi))
        if not mask.any():
            continue
        acc = correct[mask].mean()
        c = conf[mask].mean()
        gap = acc - c
        total += mask.mean() * abs(gap)
        bins_out.append((lo, hi, int(mask.sum()), float(c), float(acc), float(gap)))
    return float(total), bins_out

def ace_equal_frequency(y_true, probs, n_bins=10):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == y_true).astype(float)
    order = np.argsort(conf)
    total = 0.0
    for idx in np.array_split(order, n_bins):
        if len(idx):
            total += (len(idx)/len(y_true)) * abs(correct[idx].mean() - conf[idx].mean())
    return float(total)

def binary_brier(y_true, probs):
    return float(np.mean((probs[:,1] - y_true) ** 2))

def threshold_stats(y_true, probs, tau):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    covered = conf >= tau
    return {
        f"HCE@{tau:.2f}": int(((pred != y_true) & covered).sum()),
        f"Coverage@{tau:.2f}": float(covered.mean()),
        f"SelectiveError@{tau:.2f}": float((pred[covered] != y_true[covered]).mean()) if covered.any() else np.nan,
    }

def compute_metrics(y_true, probs):
    pred = probs.argmax(axis=1)
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "brier": binary_brier(y_true, probs),
        "nll": log_loss(y_true, probs, labels=[0,1]),
    }
    out["ece"], _ = ece_equal_width(y_true, probs, 10)
    out["ace"] = ace_equal_frequency(y_true, probs, 10)
    for tau in [0.80, 0.90, 0.95]:
        out.update(threshold_stats(y_true, probs, tau))
    return out


## 9. Training and inference engine

In [ ]:
def make_loader(df, batch_size, shuffle, include_agreement=False, seed=42):
    ds = TextDataset(df, include_agreement=include_agreement)
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        collate_fn=collate_batch, generator=g if shuffle else None
    )

def train_model(train_df, seed, smoothing_type="none", epsilon=0.0, save_dir=None):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name, num_labels=2
    ).to(device)

    loader = make_loader(
        train_df, CFG.train_batch_size, True,
        include_agreement=(smoothing_type == "aals"), seed=seed
    )

    no_decay = ["bias", "LayerNorm.weight"]
    grouped = [
        {"params":[p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay":CFG.weight_decay},
        {"params":[p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay":0.0},
    ]
    optimizer = torch.optim.AdamW(grouped, lr=CFG.learning_rate)
    total_steps = len(loader) * CFG.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, CFG.warmup_steps, total_steps
    )

    for epoch in range(CFG.epochs):
        model.train()
        running = 0.0
        for batch in loader:
            labels = batch.pop("labels").to(device)
            agreement = batch.pop("agreement", None)
            if agreement is not None:
                agreement = agreement.to(device)
            inputs = {k:v.to(device) for k,v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            logits = model(**inputs).logits

            if smoothing_type == "none":
                loss = F.cross_entropy(logits, labels)
            elif smoothing_type == "fixed":
                loss = soft_target_cross_entropy(logits, labels, epsilon)
            elif smoothing_type == "aals":
                loss = aals_cross_entropy(logits, labels, agreement, epsilon, A_MIN)
            else:
                raise ValueError(smoothing_type)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()
            scheduler.step()
            running += float(loss.detach().cpu())

        print(f"seed={seed} epoch={epoch+1}/{CFG.epochs} loss={running/len(loader):.6f}")

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
    return model

@torch.no_grad()
def predict_logits(model, df):
    loader = make_loader(df, CFG.eval_batch_size, False)
    model.eval()
    logits_all, labels_all = [], []
    for batch in loader:
        labels_all.append(batch.pop("labels").numpy())
        inputs = {k:v.to(device) for k,v in batch.items()}
        logits_all.append(model(**inputs).logits.detach().cpu().numpy())
    return np.concatenate(logits_all), np.concatenate(labels_all)

def logits_to_probs(logits, temperature=1.0):
    return softmax(logits / temperature, axis=1)


## 10. Fit-only candidate training and validation-only selection

Candidates:
- Vanilla
- LS 0.05
- LS 0.10
- LS 0.15
- optionally AALS 0.10 / 0.15 after its missing original parameters are recovered.

**Do not evaluate the official test until the lock file is written.**


In [ ]:
CANDIDATES = [
    {"code":"M0", "name":"Vanilla", "type":"none", "epsilon":0.00},
    {"code":"M1_005", "name":"LS 0.05", "type":"fixed", "epsilon":0.05},
    {"code":"M1_010", "name":"LS 0.10", "type":"fixed", "epsilon":0.10},
    {"code":"M1_015", "name":"LS 0.15", "type":"fixed", "epsilon":0.15},
]

if RUN_AALS:
    if AGREEMENT_COL is None or A_MIN is None:
        raise ValueError("RUN_AALS=True requires verified AGREEMENT_COL and A_MIN")
    CANDIDATES += [
        {"code":"M2_010", "name":"AALS 0.10", "type":"aals", "epsilon":0.10},
        {"code":"M2_015", "name":"AALS 0.15", "type":"aals", "epsilon":0.15},
    ]

rows = []
for c in CANDIDATES:
    print("\n===", c["name"], "===")
    model = train_model(
        fit_df, PRIMARY_SEED, c["type"], c["epsilon"],
        MODEL_DIR / "e1_candidates" / c["code"]
    )
    logits, y = predict_logits(model, val_df)
    probs = logits_to_probs(logits)
    met = compute_metrics(y, probs)
    np.save(ARTIFACT_DIR / f"{c['code']}_validation_logits.npy", logits)
    rows.append({**c, **met})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

val_results = pd.DataFrame(rows)
val_results.to_csv(RESULT_DIR / "e1_validation_candidates.csv", index=False)
display(val_results[["code","name","f1","brier","nll","ece","ace"]])

ranked = val_results.sort_values(
    ["ece","brier","f1"], ascending=[True,True,False], kind="mergesort"
).reset_index(drop=True)
selected = ranked.iloc[0].to_dict()
print("Selected:", selected["name"], selected["ece"], selected["brier"], selected["f1"])


## 11. Evidence lock

The submitted study selected LS ε=0.10. If this run selects another model, stop and reconcile the split/environment/code before opening the official test.


In [ ]:
lock = {
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "split_source": split_source,
    "selection_rule": ["lowest_ece","lowest_brier","highest_f1"],
    "selected_code": selected["code"],
    "selected_name": selected["name"],
    "selected_epsilon": selected["epsilon"],
    "validation_metrics": {k:float(selected[k]) for k in ["f1","brier","nll","ece","ace"]},
    "official_test_consulted": False,
}
lock_path = ARTIFACT_DIR / "model_selection_lock.json"
lock_path.write_text(json.dumps(lock, indent=2))
print(lock_path.read_text())

if selected["name"] != "LS 0.10":
    raise RuntimeError("Canonical APIN study selected LS 0.10. Reconcile before test evaluation.")


## 12. Final retraining on fit + validation; calibration remains untouched

In [ ]:
train_final = pd.concat([fit_df, val_df], ignore_index=True)
assert len(train_final) == 2415
assert len(cal_df) == 423

final_logits_cal = {}
final_logits_test = {}
official_rows = []

for c in CANDIDATES:
    print("\nFINAL ===", c["name"], "===")
    model = train_model(
        train_final, PRIMARY_SEED, c["type"], c["epsilon"],
        MODEL_DIR / "e1_final" / c["code"]
    )

    cal_logits, cal_y = predict_logits(model, cal_df)
    test_logits, test_y = predict_logits(model, test)

    np.save(ARTIFACT_DIR / f"{c['code']}_calibration_logits.npy", cal_logits)
    np.save(ARTIFACT_DIR / f"{c['code']}_official_test_logits.npy", test_logits)

    met = compute_metrics(test_y, logits_to_probs(test_logits))
    official_rows.append({**c, **met})
    final_logits_cal[c["code"]] = cal_logits
    final_logits_test[c["code"]] = test_logits

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

official_results = pd.DataFrame(official_rows)
official_results.to_csv(RESULT_DIR / "e1_official_test_scenarios.csv", index=False)
display(official_results[["code","name","accuracy","precision","recall","f1","brier","nll","ece","ace","HCE@0.90","Coverage@0.90"]])


## 13. Temperature scaling on the untouched calibration subset

Submitted diagnostic targets:
- T = **0.952997**
- calibration NLL: **0.384907 → 0.384483**


In [ ]:
selected_code = "M1_010"
cal_logits = final_logits_cal[selected_code]
test_logits_selected = final_logits_test[selected_code]

def nll_at_temperature(T, logits, y):
    if T <= 0:
        return np.inf
    return log_loss(y, logits_to_probs(logits, T), labels=[0,1])

before_cal_nll = nll_at_temperature(1.0, cal_logits, cal_y)
opt = minimize_scalar(
    lambda t: nll_at_temperature(t, cal_logits, cal_y),
    bounds=(0.05,10.0), method="bounded", options={"xatol":1e-12}
)
T_hat = float(opt.x)
after_cal_nll = float(opt.fun)

print("T_hat:", T_hat)
print("Calibration NLL before:", before_cal_nll)
print("Calibration NLL after:", after_cal_nll)

ts_probs = logits_to_probs(test_logits_selected, T_hat)
ts_metrics = compute_metrics(test_y, ts_probs)
ts_row = {
    "code":"M3", "name":"LS 0.10 + TS",
    "type":"temperature_scaling", "epsilon":0.10,
    "temperature":T_hat, **ts_metrics
}
pd.DataFrame([ts_row]).to_csv(RESULT_DIR / "e1_temperature_scaling.csv", index=False)


## 14. Numerical reproduction audit against the submitted APIN manuscript

This is a diagnostic comparison only. Never tune the test pipeline to chase these numbers.


In [ ]:
TARGETS = {
    "M0": {
        "accuracy":0.7958, "precision":0.7753, "recall":0.8509, "f1":0.8114,
        "brier":0.1658, "nll":0.5920, "ece":0.1286, "ace":0.1298, "HCE@0.90":76
    },
    "M1_010": {
        "accuracy":0.7972, "precision":0.7828, "recall":0.8401, "f1":0.8105,
        "brier":0.1478, "nll":0.4622, "ece":0.0421, "ace":0.0601, "HCE@0.90":26
    },
    "M3": {
        "brier":0.1487, "nll":0.4655, "ece":0.0508, "ace":0.0656, "HCE@0.90":29
    }
}

result_map = {r["code"]:r for r in official_results.to_dict("records")}
result_map["M3"] = ts_row

audit_rows = []
for code_, target_metrics in TARGETS.items():
    got = result_map[code_]
    for metric, target in target_metrics.items():
        observed = got[metric]
        if metric.startswith("HCE"):
            diff = abs(int(observed) - int(target))
            ok = diff == 0
        else:
            diff = abs(float(observed) - float(target))
            ok = diff <= REPRO_ABS_TOL
        audit_rows.append({
            "code":code_, "metric":metric, "target":target,
            "observed":observed, "abs_diff":diff,
            "within_diagnostic_tolerance":ok
        })

repro_audit = pd.DataFrame(audit_rows)
repro_audit.to_csv(RESULT_DIR / "e1_reproduction_audit.csv", index=False)
display(repro_audit)
print("All diagnostic checks passed:", bool(repro_audit["within_diagnostic_tolerance"].all()))


## 15. Confusion matrices and exact McNemar test

Submitted targets:
- M0: TN=255, FP=91, FN=55, TP=314
- LS 0.10: TN=260, FP=86, FN=59, TP=310
- discordant pairs: b=9, c=8; p=1.000


In [ ]:
m0_logits = final_logits_test["M0"]
ls_logits = final_logits_test["M1_010"]
m0_pred = m0_logits.argmax(axis=1)
ls_pred = ls_logits.argmax(axis=1)

print("M0 CM:\n", confusion_matrix(test_y, m0_pred))
print("LS CM:\n", confusion_matrix(test_y, ls_pred))

m0_correct = m0_pred == test_y
ls_correct = ls_pred == test_y
b = int((m0_correct & ~ls_correct).sum())
c = int((~m0_correct & ls_correct).sum())
n_disc = b + c
p_exact = 1.0 if n_disc == 0 else binomtest(min(b,c), n=n_disc, p=0.5, alternative="two-sided").pvalue

print("b =", b, "c =", c, "p =", p_exact)
(ARTIFACT_DIR / "mcnemar_seed42.json").write_text(
    json.dumps({"b":b, "c":c, "p_exact":float(p_exact)}, indent=2)
)


## 16. 10,000 paired instance bootstraps

This reproduces the submitted inference procedure. E7 will later upgrade it to a `post_id` cluster bootstrap.


In [ ]:
def paired_bootstrap(y_true, probs_a, probs_b, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    metrics = ["accuracy","precision","recall","f1","brier","nll","ece","ace","HCE@0.90"]
    rows = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ya = y_true[idx]
        ma = compute_metrics(ya, probs_a[idx])
        mb = compute_metrics(ya, probs_b[idx])
        rows.append({m:mb[m]-ma[m] for m in metrics})
    boot = pd.DataFrame(rows)

    summary = []
    for m in metrics:
        q = boot[m].quantile([0.025,0.5,0.975])
        summary.append({
            "metric":m,
            "bootstrap_mean_delta_LS_minus_M0":boot[m].mean(),
            "ci_2.5":q.loc[0.025],
            "median":q.loc[0.5],
            "ci_97.5":q.loc[0.975]
        })
    return boot, pd.DataFrame(summary)

m0_probs = logits_to_probs(m0_logits)
ls_probs = logits_to_probs(ls_logits)
boot_raw, boot_summary = paired_bootstrap(test_y, m0_probs, ls_probs, 10000, PRIMARY_SEED)
boot_summary.to_csv(RESULT_DIR / "e1_paired_bootstrap_summary.csv", index=False)
display(boot_summary)


## 17. Optional five-seed paired reproduction

Enable `RUN_MULTISEED=True` only after seed-42 reproduction is reconciled.


In [ ]:
if RUN_MULTISEED:
    seed_rows = []
    for seed in PAIRED_SEEDS:
        for code_, name_, typ_, eps_ in [
            ("M0","Vanilla","none",0.0),
            ("M1_010","LS 0.10","fixed",0.10),
        ]:
            print(f"seed={seed} {name_}")
            model = train_model(train_final, seed, typ_, eps_, None)
            logits, y = predict_logits(model, test)
            met = compute_metrics(y, logits_to_probs(logits))
            seed_rows.append({"seed":seed,"code":code_,"name":name_,**met})
            np.save(ARTIFACT_DIR / f"{code_}_test_logits_seed{seed}.npy", logits)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    seed_df = pd.DataFrame(seed_rows)
    seed_df.to_csv(RESULT_DIR / "e1_five_seed_results.csv", index=False)
    metrics_ = ["accuracy","precision","recall","f1","brier","nll","ece","ace","HCE@0.90","Coverage@0.90"]
    display(seed_df.groupby(["code","name"])[metrics_].agg(["mean","std"]))
else:
    print("RUN_MULTISEED=False — skipped.")


## 18. Export prediction-level evidence package for E2/E5

In [ ]:
export = test[[ID_COL, POST_COL, TEXT_COL, LABEL_COL]].copy()

for code_ in ["M0","M1_010"]:
    logits = final_logits_test[code_]
    probs = logits_to_probs(logits)
    export[f"{code_}_logit0"] = logits[:,0]
    export[f"{code_}_logit1"] = logits[:,1]
    export[f"{code_}_p0"] = probs[:,0]
    export[f"{code_}_p1"] = probs[:,1]
    export[f"{code_}_pred"] = probs.argmax(axis=1)
    export[f"{code_}_confidence"] = probs.max(axis=1)

export["M3_p0"] = ts_probs[:,0]
export["M3_p1"] = ts_probs[:,1]
export["M3_pred"] = ts_probs.argmax(axis=1)
export["M3_confidence"] = ts_probs.max(axis=1)

export.to_csv(ARTIFACT_DIR / "predictions_seed42.csv", index=False)
print("Saved:", ARTIFACT_DIR / "predictions_seed42.csv")


## 19. E1 completion gate

Do **not** proceed to E2 until these are resolved:

### Required
- [ ] development = 2,838
- [ ] official test = 715
- [ ] `id` overlap = 0
- [ ] `post_id` overlap = 0
- [ ] exact cross-split text duplicates = 3
- [ ] fit/validation/calibration = 1,994 / 421 / 423
- [ ] class counts = 942/1052, 203/218, 205/218
- [ ] validation selects LS ε=0.10
- [ ] seed-42 metrics are reconciled with APIN
- [ ] temperature fit is reconciled
- [ ] lock file exists before test evaluation

### AALS-specific
- [ ] original annotator-agreement field recovered
- [ ] original `a_min` recovered
- [ ] original AALS implementation checked against Eq. (4)–(5)

### Strongly preferred
- [ ] original split manifest recovered
- [ ] original environment/package lock recovered
- [ ] five-seed means reproduced

After E1 passes, freeze `artifacts/e1/` and use its logits/predictions as the input to **E2 — Strong Calibration Baseline Benchmark**.
